# Enterprise Cloud ETL Pipeline: Marketing & E-Commerce Analytics
### Domain: Retail & Marketing Analytics | Tech Stack: Python, AWS S3, SQL (Athena), Power BI
**Author:** Utkarsh (Data Analyst)

**Project Overview:**
This notebook automates an end-to-end data pipeline. It extracts product feeds from a live Web API, transforms the data for Marketing KPIs, scales the dataset to 10,000+ transactional records using simulation loops, and loads it securely into an AWS S3 Data Lake for cloud SQL querying.

In [9]:
!pip install boto3 openpyxl

In [11]:
# Step 1: Importing required libraries  
import pandas as pd
import requests
import boto3
import random
from io import StringIO
from datetime import datetime, timedelta
print("✅ Data Analytics Environments & Libraries Successfully Loaded.")

✅ Data Analytics Environments & Libraries Successfully Loaded.


In [14]:
# Step 2: Extracting Live Product Feed via Web API
url = "https://fakestoreapi.com/products"
response = requests.get(url)

if response.status_code == 200:
    api_products = response.json()
    print(f"📥 [EXTRACT SUCCESS] Fetched {len(api_products)} base product schemas from live API endpoint.")
else:
    print(f"❌ [EXTRACT ERROR] API Request failed. Status Code: {response.status_code}")
    exit()

📥 [EXTRACT SUCCESS] Fetched 20 base product schemas from live API endpoint.


In [15]:
# Step 3: Transforming Data & Scaling to 10000+ transactions(Big Data Simulation)
print("⚙️ [TRANSFORM STARTED] Synthesizing retail transactions and applying marketing tags...")

large_sales_data = []
order_id_counter = 100001

# Marketing dimensions for demographic and cohort analysis
countries = ['India', 'USA', 'UK', 'Canada', 'Germany', 'Australia']
payment_methods = ['Credit Card', 'UPI', 'PayPal', 'Net Banking']

# Generating 10,500 high-volume rows for cloud-scale analytics
for i in range(10500):
    # Pick a random original product from the API
    prod = random.choice(api_products)
    
    # Simulating purchase dates for the last 6 months
    random_days = random.randint(0, 180)
    order_date = (datetime.now() - timedelta(days=random_days)).strftime('%Y-%m-%d')
    
    quantity = random.randint(1, 5)
    price_inr = round(prod['price'] * 84, 2)  # Currency Normalization: USD to INR
    total_sales_inr = round(price_inr * quantity, 2)
    
    rating_score = prod['rating']['rate'] if 'rating' in prod else 4.0

    # Structuring Corporate Grade Data Schema
    transaction = {
        'Order_ID': f"TXN-{order_id_counter}",
        'Order_Date': order_date,
        'Product_ID': prod['id'],
        'Product_Name': prod['title'],
        'Category': prod['category'],
        'Price_INR': price_inr,
        'Quantity': quantity,
        'Total_Sales_INR': total_sales_inr,
        'Customer_Country': random.choice(countries),
        'Payment_Method': random.choice(payment_methods),
        'Product_Rating': rating_score,
        'Pipeline_Run_Time': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    }
    
    large_sales_data.append(transaction)
    order_id_counter += 1

# Converting processed list into a highly structured Pandas DataFrame
df_marketing_final = pd.DataFrame(large_sales_data)

print(f"✅ [TRANSFORM COMPLETE] Data scaled successfully. Shape: {df_marketing_final.shape}")
# Displaying top 3 rows for data quality audit
df_marketing_final.head(3)

⚙️ [TRANSFORM STARTED] Synthesizing retail transactions and applying marketing tags...
✅ [TRANSFORM COMPLETE] Data scaled successfully. Shape: (10500, 12)


,Order_ID,Order_Date,Product_ID,Product_Name,Category,Price_INR,Quantity,Total_Sales_INR,Customer_Country,Payment_Method,Product_Rating,Pipeline_Run_Time
0,TXN-100001,2026-03-11,18,MBJ Women's Solid Short Sleeve Boat Neck V,women's clothing,827.40,2,1654.80,USA,Credit Card,4.7,2026-06-16 09:30:24
1,TXN-100002,2026-02-21,18,MBJ Women's Solid Short Sleeve Boat Neck V,women's clothing,827.40,5,4137.00,India,Net Banking,4.7,2026-06-16 09:30:24
2,TXN-100003,2026-03-07,15,BIYLACLESEN Women's 3-in-1 Snowboard Jacket Wi...,women's clothing,4787.16,3,14361.48,Australia,UPI,2.6,2026-06-16 09:30:24


In [17]:
# Step 4: Loading Structured Dataset into AWS S3 Data Lake
# ⚠️ Configuration: Input your AWS Identity Keys and Bucket details here

AWS_ACCESS_KEY = "ACCESS KEY"
AWS_SECRET_KEY = "SECRET KEY"
BUCKET_NAME = "utkarsh-marketing-sales-data" 
FILE_NAME = "big_marketing_sales_data.csv"

print(f"📤 [LOAD STARTED] Establishing secure connection and uploading to S3: {BUCKET_NAME}...")

try:
    # Initializing the AWS Boto3 S3 Client
    s3_client = boto3.client(
        's3',
        aws_access_key_id=AWS_ACCESS_KEY,
        aws_secret_access_key=AWS_SECRET_KEY
    )
    
    # Converting DataFrame directly into CSV string in volatile memory (efficient streaming)
    csv_buffer = StringIO()
    df_marketing_final.to_csv(csv_buffer, index=False)
    
    # Pushing object to AWS Cloud
    s3_client.put_object(
        Bucket=BUCKET_NAME,
        Key=FILE_NAME,
        Body=csv_buffer.getvalue()
    )
    print("🎉 [ETL JOB SUCCESSFUL] 10,500+ rows of original marketing data are now live on AWS S3 Cloud!")

except Exception as e:
    print(f"❌ [AWS CLOUD ERROR] Upload failed: {e}")

📤 [LOAD STARTED] Establishing secure connection and uploading to S3: utkarsh-marketing-sales-data...
🎉 [ETL JOB SUCCESSFUL] 10,500+ rows of original marketing data are now live on AWS S3 Cloud!


In [1]:
!pip install pandas boto3 matplotlib